# Proportion of Residential Area Analysis

This notebook analyzes residential area proportions in Jakarta using geospatial data and machine learning classification.

**Analysis Pipeline:**
1. Load administrative boundaries (Kelurahan & RW) and building footprints
2. Extract building heights and NDVI from raster data
3. Calculate building and spatial features (height, shape regularity, density, green ratio)
4. Classify residential types using hierarchical decision rules
5. Visualize and export results

## Load Data

### Kelurahan (Neighbourhood)

In [ ]:
# ──────────────────────────── IMPORTS ────────────────────────────

import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import logging
import os
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.stats import circvar

# ──────────────────────────── CONFIG ────────────────────────────

# File paths
KELURAHAN_SHP_PATH = "../input/adm_dki-jakarta_kelurahan.shp"
RW_SHP_PATH = "../input/Administrasi RW Jakarta.shp"
BUILDINGS_GDF_PATH = "../input/residential_area.shp"
BUILDINGS_HEIGHT_PATH = "../input/building_height_2023.tif"
NDVI_PATH = "../input/NDVI_2023.tif"
OUTPUT_DIR = "../output"

# CRS
PROJECTED_CRS = "EPSG:32748"  # UTM Zone 48S

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
log = logging.getLogger(__name__)

In [ ]:
# Load Kelurahan (neighbourhood) data
kelurahan_gdf = gpd.read_file(KELURAHAN_SHP_PATH)
kelurahan_gdf = kelurahan_gdf[['OBJECTID', 'WADMKD', 'WADMKC', 'WADMKK', 'geometry']]
kelurahan_gdf = kelurahan_gdf.to_crs(PROJECTED_CRS)

log.info(f"Kelurahan data loaded: {len(kelurahan_gdf)} features")
kelurahan_gdf.head()

### RW (Block Unit)

In [ ]:
# Load RW (Rukun Warga / block unit) data
block_unit = gpd.read_file(RW_SHP_PATH)
block_unit = block_unit[['OBJECTID', 'WADMRW', 'WADMKD', 'WADMKC', 'WADMKK', 'SHAPE_Area', 'geometry']]

# Exclude Kepulauan Seribu (Thousand Islands)
block_unit = block_unit[block_unit['WADMKK'] != 'KAB.ADM.KEP.SERIBU'].reset_index(drop=True)

# Create unique block identifier
block_unit['block'] = block_unit['WADMKD'].astype(str) + ' RW ' + block_unit['WADMRW'].astype(str)

# Fix invalid geometries
block_unit["geometry"] = block_unit["geometry"].buffer(0)

log.info(f"Block unit data loaded: {len(block_unit)} blocks")
block_unit.head()

### Building Footprint

In [ ]:
# Load building footprints
buildings_gdf = gpd.read_file(BUILDINGS_GDF_PATH)
buildings_gdf = buildings_gdf[['id', 'osmid', 'class', 'type', 'name', 'area', 'geometry']]

log.info(f"Building footprints loaded: {len(buildings_gdf)} buildings")
buildings_gdf.head()

## Data Processing

### Spatial Join

In [ ]:
# Assign each building to its corresponding block unit
buildings_gdf['block'] = gpd.sjoin(buildings_gdf, block_unit, how='left', predicate='within')['block']

log.info(f"Buildings assigned to blocks: {buildings_gdf['block'].notnull().sum()}")
log.info(f"Buildings without block assignment: {buildings_gdf['block'].isnull().sum()}")

In [ ]:
# Preview buildings with block assignments
buildings_gdf[['id', 'class', 'type', 'area', 'block']].head()

### Extract Raster Values

#### Building Heights

In [ ]:
# Sample building height values from raster at building centroids
with rasterio.open(BUILDINGS_HEIGHT_PATH) as src:
    if buildings_gdf.crs != src.crs:
        buildings_gdf = buildings_gdf.to_crs(src.crs)
    coords = [(geom.centroid.x, geom.centroid.y) for geom in buildings_gdf.geometry]
    vals = [round(v[0], 1) for v in src.sample(coords)]

buildings_gdf['height'] = vals
log.info("Building heights extracted from raster")

#### NDVI

In [ ]:
# Sample NDVI values from raster at building centroids
with rasterio.open(NDVI_PATH) as src:
    if buildings_gdf.crs != src.crs:
        buildings_gdf = buildings_gdf.to_crs(src.crs)
    coords = [(geom.centroid.x, geom.centroid.y) for geom in buildings_gdf.geometry]
    vals = [v[0] for v in src.sample(coords)]

buildings_gdf['ndvi'] = vals
log.info("NDVI values extracted from raster")

## Feature Engineering

### 1. Building Height Classification

In [ ]:
# Estimate number of floors from building height
def estimate_floors(height_m):
    if np.isnan(height_m):
        return np.nan
    elif height_m <= 3:
        return 1
    elif height_m <= 7:
        return 2
    elif height_m <= 10:
        return 3
    else:
        return round(height_m / 3.4)

buildings_gdf['floors'] = buildings_gdf['height'].apply(estimate_floors)

# Classify buildings as low or middle-to-high rise (14 floors ≈ 50m threshold)
buildings_gdf['bld_height'] = np.where(buildings_gdf['floors'] > 14, 'middle-to-high', 'low')

log.info(f"Buildings classified: {(buildings_gdf['bld_height'] == 'middle-to-high').sum()} high-rise, "
         f"{(buildings_gdf['bld_height'] == 'low').sum()} low-rise")

In [ ]:
# Aggregate building height to block level (majority class)
block_bld_height = buildings_gdf.groupby('block')['bld_height'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else np.nan
)
block_unit = block_unit.merge(block_bld_height.rename('bld_height'), 
                                left_on='block', right_index=True, how='left')

### 2. Building Shape & Layout Regularity

In [ ]:
# Convert to projected CRS if needed for accurate geometric calculations
if block_unit.crs != PROJECTED_CRS:
    block_unit = block_unit.to_crs(PROJECTED_CRS)
if buildings_gdf.crs != PROJECTED_CRS:
    buildings_gdf = buildings_gdf.to_crs(PROJECTED_CRS)

In [ ]:
# Helper functions for shape analysis
def get_orientation(polygon):
    """Get building orientation from minimum rotated rectangle"""
    min_rect = polygon.minimum_rotated_rectangle
    coords = list(min_rect.exterior.coords)
    edge1 = np.array(coords[1]) - np.array(coords[0])
    angle = np.arctan2(edge1[1], edge1[0]) * 180 / np.pi
    return angle % 180

def compactness(polygon):
    """Calculate compactness as area ratio to minimum bounding rectangle"""
    area = polygon.area
    min_rect = polygon.minimum_rotated_rectangle
    rect_area = min_rect.area
    return area / rect_area if rect_area > 0 else 0

# Compute shape features for each block
block_features = []
for blk, grp in buildings_gdf.groupby('block'):
    orientations = grp.geometry.apply(get_orientation)
    orient_var = circvar(np.deg2rad(orientations * 2)) / 4
    compact_mean = grp.geometry.apply(compactness).mean()
    
    centroids = grp.geometry.centroid
    inter_dist = centroids.apply(lambda x: centroids.distance(x).mean()).mean()
    
    blk_geom = block_unit.loc[block_unit['block'] == blk, 'geometry'].iloc[0]
    shape_index = blk_geom.length**2 / (4 * np.pi * blk_geom.area)
    
    block_features.append({
        'block': blk,
        'orient': orient_var,
        'compact': compact_mean,
        'inter_dist': inter_dist,
        'shape_index': shape_index
    })

building_shape = pd.DataFrame(block_features)

In [ ]:
# Classify shape regularity using K-Means clustering
cols = ['orient', 'compact', 'inter_dist', 'shape_index']
X = building_shape[cols].fillna(building_shape[cols].median())

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=2, random_state=0, n_init=10)
labels = kmeans.fit_predict(Xs)
building_shape['reg_cluster'] = labels

# Determine which cluster represents "regular" layout
centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=cols)
centroids['score'] = (centroids['compact'] - centroids['orient'] - 
                      centroids['shape_index'] / (centroids['shape_index'].max() + 1e-9))
best_label = centroids['score'].idxmax()

mapping = {best_label: 'regular'}
mapping.update({lbl: 'irregular' for lbl in set(labels) if lbl != best_label})
building_shape['bld_shape'] = building_shape['reg_cluster'].map(mapping)

# Merge to block_unit
block_unit = block_unit.merge(building_shape[['block', 'bld_shape']], on='block', how='left')
log.info("Block shape regularity classified")

### 3. Building Density

In [ ]:
# Calculate building density metrics for each block
building_density_data = []
for idx, block in block_unit.iterrows():
    buildings = buildings_gdf[buildings_gdf.geometry.within(block.geometry)]
    total_building_area = buildings.geometry.area.sum()
    building_count = len(buildings)
    block_area = block.geometry.area
    
    building_density_data.append({
        'block': block['block'],
        'building_area_density': total_building_area / block_area if block_area > 0 else 0,
        'building_count_density': building_count / block_area if block_area > 0 else 0
    })

building_density = pd.DataFrame(building_density_data)

In [ ]:
# Classify density using K-Means clustering
cols = ['building_area_density', 'building_count_density']
X = building_density[cols].fillna(building_density[cols].median())

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=2, random_state=0, n_init=10)
labels = kmeans.fit_predict(Xs)
building_density['dense_cluster'] = labels

# Determine which cluster represents higher density
centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=cols)
centroids['score'] = centroids['building_area_density'] + centroids['building_count_density']
best_label = centroids['score'].idxmax()

mapping = {best_label: 'middle-to-high'}
mapping.update({lbl: 'low-to-middle' for lbl in set(labels) if lbl != best_label})
building_density['bld_density'] = building_density['dense_cluster'].map(mapping)

# Merge to block_unit
block_unit = block_unit.merge(building_density[['block', 'bld_density']], on='block', how='left')
log.info("Building density classified")

### 4. Green Ratio (NDVI-based)

In [ ]:
# Classify green ratio using K-Means clustering
cols = ['ndvi']
X = buildings_gdf[cols].fillna(buildings_gdf[cols].median())

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=2, random_state=0, n_init=10)
labels = kmeans.fit_predict(Xs)
buildings_gdf['green_cluster'] = labels

# Determine which cluster represents higher vegetation
centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=cols)
centroids['score'] = centroids['ndvi']
best_label = centroids['score'].idxmax()

mapping = {best_label: 'high'}
mapping.update({lbl: 'low' for lbl in set(labels) if lbl != best_label})
buildings_gdf['green_ratio'] = buildings_gdf['green_cluster'].map(mapping)

log.info(f"Green ratio classified: {(buildings_gdf['green_ratio'] == 'high').sum()} high, "
         f"{(buildings_gdf['green_ratio'] == 'low').sum()} low")

In [ ]:
# Aggregate green ratio to block level (majority class)
block_green_ratio = buildings_gdf.groupby('block')['green_ratio'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else np.nan
)
block_unit = block_unit.merge(block_green_ratio.rename('green_ratio'), 
                                left_on='block', right_index=True, how='left')

## Residential Type Classification

In [ ]:
# Classify residential type based on combined features
def classify_residential(row):
    """
    Hierarchical classification:
    1. Height → High-rise
    2. Shape → Regular
    3. Density + Green → Rural/Urban Village
    """
    hc = row.get('bld_height')
    sc = row.get('bld_shape')
    dc = row.get('bld_density')
    gc = row.get('green_ratio')
    
    if pd.notna(hc) and hc == 'middle-to-high':
        return 'High-rise'
    
    if pd.notna(sc) and sc == 'regular':
        return 'Regular'
    
    if pd.notna(dc) and dc == 'low-to-middle':
        if pd.notna(gc) and gc == 'high':
            return 'Rural'
        else:
            return 'Urban Village'
    
    if pd.notna(gc):
        return 'Rural' if gc == 'high' else 'Urban Village'
    
    return np.nan

block_unit['res_type'] = block_unit.apply(classify_residential, axis=1)
res_type_map = {'High-rise': 0, 'Regular': 1, 'Urban Village': 3, 'Rural': 4}
block_unit['res_value'] = block_unit['res_type'].map(res_type_map)

log.info(f"Residential types:\n{block_unit['res_type'].value_counts()}")

In [ ]:
# View final results
block_unit[['block', 'bld_height', 'bld_shape', 'bld_density', 'green_ratio', 'res_type']].head(10)

## Visualization

In [ ]:
# Plot residential types by block
fig, ax = plt.subplots(figsize=(12, 10))
block_unit.plot(ax=ax, column='res_type', categorical=True, legend=True, 
                cmap='viridis', legend_kwds={'loc': 'lower left'}, alpha=0.8)
kelurahan_gdf.boundary.plot(ax=ax, color='white', linewidth=0.5, alpha=0.5)
ax.set_title('Residential Type Classification by Block (RW)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## Export Results

In [ ]:
# Create output directory and save results
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save block units with residential classification
block_unit.to_file(os.path.join(OUTPUT_DIR, "residential_type.shp"))
log.info(f"Block units saved to {OUTPUT_DIR}/residential_type.shp")

# Save buildings with all features
buildings_gdf.to_file(os.path.join(OUTPUT_DIR, "buildings.shp"))
log.info(f"Buildings saved to {OUTPUT_DIR}/buildings.shp")

log.info("Processing complete!")